# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danielajetunmobi/flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. The target's baseline — settled here, because ML-06 could not settle it

Two different things get called a baseline in this assignment. Section 1 onward builds the **rule
baseline**: a hand-written score the model must beat. This section settles the other one first,
because nothing below can be evaluated without it.

**What ML-06 established.** `target = asinh(future_daily) − asinh(baseline_daily)` used the recent
30-day rate as `baseline_daily`. That quantity also drives the features, so a page with a large recent
window scores high on the signals *and* low on the target by arithmetic alone. Three separate
findings turned out to be that one mechanism:

| finding | what it looked like | what it was |
|---|---|---|
| Test 3 | a 7x decline gradient by peak ratio | 89% the label's own exclusion rule |
| simulation | pages reverting after a peak | a null with no future produced 79.0 points against 12.6 observed |
| Test 5 → Test 10 | signal hiding inside clients | a null produced ρ −0.66 against −0.09 observed |

**So the baseline must not appear in the features.** That is a testable property, not a matter of
taste, and the test already exists: build the target on a candidate baseline, then correlate the
signals against it using a **randomised future**. A null carrying no information about what happens
next should produce **ρ ≈ 0**. Any candidate where it does not is manufacturing the correlation.

In [1]:
%pip install -q duckdb huggingface_hub pandas numpy scipy matplotlib python-dotenv

import os
import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

token = os.environ.get("HF_TOKEN")
if not token:
    import getpass
    token = getpass.getpass("Hugging Face token (read): ")

REPO = "FlyRank/internship-warehouse"
MONTHS = ["2025-12", "2026-01", "2026-02", "2026-03", "2026-04", "2026-05", "2026-06"]
daily_files = [
    hf_hub_download(repo_id=REPO, repo_type="dataset",
                    filename=f"fact_content_daily_performance/month={m}/data_0.parquet", token=token)
    for m in MONTHS
]
con = duckdb.connect()
REL = "read_parquet([" + ", ".join(f"'{f}'" for f in daily_files) + "])"
D1 = "2026-03-31"

# Daily series for pages with enough pre-decision history to hold out a window.
q = f"""
WITH dense AS (
  SELECT content_hash_id FROM {REL}
  WHERE report_date < DATE '{D1}'
  GROUP BY 1 HAVING COUNT(*) FILTER (WHERE gsc_impressions > 0) >= 120)
SELECT f.content_hash_id, f.report_date, f.gsc_impressions
FROM {REL} f JOIN dense USING (content_hash_id)
WHERE f.report_date < DATE '{D1}' + INTERVAL 30 DAY
ORDER BY 1, 2"""
piv = con.sql(q).df().pivot(index="report_date", columns="content_hash_id",
                            values="gsc_impressions").fillna(0)
piv.index = pd.to_datetime(piv.index)

D_ts = pd.Timestamp(D1)
pre = piv[piv.index < D_ts]
fut = piv[(piv.index >= D_ts) & (piv.index < D_ts + pd.Timedelta(days=30))]
print(f"{piv.shape[1]:,} pages | {len(pre)} pre-decision days | {len(fut)} future days")


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


27,801 pages | 120 pre-decision days | 30 future days


**Four candidate baselines, chosen to span the trade-off.**

The tension is real: a baseline should reflect *where the page currently is*, which pushes toward
recent data — but recent data is what the features are built from. Each candidate resolves it
differently.

| candidate | definition | shares with features? |
|---|---|---|
| **A** recent 30d | the current design | fully — this is the control |
| **B** 90-day mean | contains the recent 30 | partly |
| **C** days 30–90 | the recent 30 held out entirely | no |
| **D** 90-day median | robust to spikes, still contains recent 30 | partly |

**C is the one to beat.** It costs a month of recency and gains complete separation. Whether that
trade is worth making is what the null decides, not preference.

In [2]:
rng = np.random.default_rng(7)

recent30 = pre.tail(30).mean().values          # most recent 30 days
older60 = pre.tail(90).head(60).mean().values  # days 30-90 back: recent month held out
full90 = pre.tail(90).mean().values
med90 = pre.tail(90).median().values
actual = fut.mean().values

# Null future: a random 30-day window from each page's OWN pre-decision history.
# Level and volatility preserved; all information about what happens next destroyed.
pre_arr = pre.values
starts = rng.integers(0, len(pre_arr) - 30, size=piv.shape[1])
null_fut = np.array([pre_arr[s:s + 30, i].mean() for i, s in enumerate(starts)])

safe = lambda n, d: np.divide(n, d, out=np.full_like(n, np.nan, dtype=float), where=d > 0)
signals = {
    "peak_ratio": safe(recent30, full90),
    "prior_trend": safe(recent30 - pre.tail(60).head(30).mean().values,
                        pre.tail(60).head(30).mean().values),
}
candidates = {"A recent 30d": recent30, "B 90-day mean": full90,
              "C days 30-90": older60, "D 90-day median": med90}

rows = []
for cname, base in candidates.items():
    t_obs = np.arcsinh(actual) - np.arcsinh(base)
    t_null = np.arcsinh(null_fut) - np.arcsinh(base)
    for sname, sig in signals.items():
        ok = np.isfinite(sig) & np.isfinite(t_obs) & np.isfinite(t_null)
        s = pd.Series(sig[ok])
        rows.append({
            "baseline": cname, "signal": sname, "n": int(ok.sum()),
            "rho_null": round(s.corr(pd.Series(t_null[ok]), method="spearman"), 4),
            "rho_observed": round(s.corr(pd.Series(t_obs[ok]), method="spearman"), 4),
        })

res = pd.DataFrame(rows)
res["artefact_share"] = (res["rho_null"].abs() /
                         res["rho_observed"].abs().replace(0, np.nan)).round(2)
print(res.to_string(index=False))
print()
print("rho_null is what a future carrying NO information produces.")
print("A baseline is independent of the features when rho_null is ~0.")

       baseline      signal     n  rho_null  rho_observed  artefact_share
   A recent 30d  peak_ratio 27801   -0.6827       -0.1485            4.60
   A recent 30d prior_trend 27801   -0.5977       -0.0409           14.61
  B 90-day mean  peak_ratio 27801   -0.2233        0.3066            0.73
  B 90-day mean prior_trend 27801   -0.2107        0.3666            0.57
   C days 30-90  peak_ratio 27801    0.1249        0.5057            0.25
   C days 30-90 prior_trend 27801    0.1058        0.5438            0.19
D 90-day median  peak_ratio 27801   -0.1986        0.3507            0.57
D 90-day median prior_trend 27801   -0.1227        0.4102            0.30

rho_null is what a future carrying NO information produces.
A baseline is independent of the features when rho_null is ~0.


**Verdict: candidate C. Holding out the recent month does not just shrink the artefact — it reverses
the finding.**

27,801 pages with ≥120 active pre-decision days:

| baseline | signal | ρ null | ρ observed | artefact share |
|---|---|---|---|---|
| **A** recent 30d | `peak_ratio` | **−0.6827** | −0.1485 | **4.60** |
| **A** recent 30d | `prior_trend` | **−0.5977** | −0.0409 | **14.61** |
| **B** 90-day mean | `peak_ratio` | −0.2233 | +0.3066 | 0.73 |
| **B** 90-day mean | `prior_trend` | −0.2107 | +0.3666 | 0.57 |
| **C** days 30–90 | `peak_ratio` | **+0.1249** | **+0.5057** | **0.25** |
| **C** days 30–90 | `prior_trend` | **+0.1058** | **+0.5438** | **0.19** |
| **D** 90-day median | `peak_ratio` | −0.1986 | +0.3507 | 0.57 |
| **D** 90-day median | `prior_trend` | −0.1227 | +0.4102 | 0.30 |

**Under A the artefact is 4.6 to 14.6 times larger than the signal. Under C the signal is 4 to 5
times larger than the artefact.** That ratio inverting is the whole point of the exercise.

**The sign flips, and that is the finding.** Under A, `peak_ratio` correlates **−0.15** with the
target: pages running hot appear to fall. Under C the same pages, the same futures, correlate
**+0.51**: pages running hot keep running above their older level. A was measuring change *from* the
hot window itself, so a hot window guaranteed a negative reading. Remove that and the relationship
points the other way.

This is consistent with everything ML-06 measured and could not interpret. Consecutive 30-day means
correlate at **0.799** — levels persist. A target built on the recent window fought that persistence;
a target built on an older window records it.

**Honest about the residual.** C's null is **+0.12** and **+0.11**, not zero. `peak_ratio` divides by
the 90-day mean, which still contains days 30–90, so a thread of shared history remains. It runs the
*same* direction as the observed signal now, so it inflates rather than inverts — and at roughly a
fifth of the magnitude. Reported rather than rounded away.

**What this settles.**

```
target = asinh(future_30d_daily_rate) - asinh(days_30_to_90_daily_rate)
```

The baseline is the page's own level over the 60 days ending one month before the decision point.
It costs a month of recency and buys separation from every feature built on the recent window.

**Caveat, same as Test 10's.** This runs on pages with ≥120 active days, where a baseline window can
actually be held out. Sparse pages have less history to spare and the residual may behave differently
there; section 2 checks the queue on the full cohort rather than assuming it carries over.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.